Este código tem o objetivo de filtrar uma base de dados contendo usuários e números de matrícula. Portanto, ao final da Pipeline, tem-se uma base de dados separada entre 'Strings' (usuários formatados com o email do inatel) e 'Numeros' (números de matrícula). Este notebook está organizado por fases, assim como a proposta do projeto, sendo a Fase 3 a versão final do Saneador, integrando as versões das fases anteriores.

## Fase 1

Dados brutos da entrada (usuários e matrículas misturados)

In [313]:
input_text = ['   Alo23',
              'natalia.batista',
              'msudid ',
              '1234556',
              '&4sss#',
              '&4sss#',
              '&4sss#',
              2034,
              2034,
              '&4sss#',
              'aline.santos',
              'anna',
              2059,
              124,
              'natalia.batista',
              1010,
              2345,
              1234567788,
              'joao.pedro',
              'maria.clara',
              123456,
              910,
              910,
              10.34]

Usando um dicionário para separar os dados por tipos

In [314]:
dicionario = {'Numeros': [], 'Strings': []}

for item in input_text:
    if isinstance(item, (int, float)):
        dicionario['Numeros'].append(item)
    else:
        dicionario['Strings'].append(item)


In [315]:
print(dicionario)

{'Numeros': [2034, 2034, 2059, 124, 1010, 2345, 1234567788, 123456, 910, 910, 10.34], 'Strings': ['   Alo23', 'natalia.batista', 'msudid ', '1234556', '&4sss#', '&4sss#', '&4sss#', '&4sss#', 'aline.santos', 'anna', 'natalia.batista', 'joao.pedro', 'maria.clara']}


Realizando um tratamento básico em strings (remoção de espaços extras, padronização de caixa, conversão de strings para números quando necessário)

In [316]:
for i in range(len(dicionario['Strings'])):
  if dicionario['Strings'][i].isdigit():
    dicionario['Strings'][i] = int(dicionario['Strings'][i])   #convertendo numeros que estão representados como strings
    #reclassificando para números no dicionário
    dicionario['Numeros'].append(dicionario['Strings'][i])
    #dicionario['Strings'].pop(i)

  else:
    dicionario['Strings'][i] = dicionario['Strings'][i].strip() #removendo espacos extras
    dicionario['Strings'][i] = dicionario['Strings'][i].lower() #padronizando todo o texto para caixa baixa
    # mantendo apenas caracteres alfanumericos nas strings (removendo caracteres especiais)
    dicionario['Strings'][i] = "".join(caractere for caractere in dicionario['Strings'][i] if caractere.isalnum() or caractere == '.')



Usando conjuntos para remover valores duplicados

In [317]:
print(dicionario)

dicionario['Strings'] = set(dicionario['Strings'])
dicionario['Strings'] = list(dicionario['Strings'])

dicionario['Numeros'] = set(dicionario['Numeros'])
dicionario['Numeros'] = list(dicionario['Numeros'])

print(dicionario)

{'Numeros': [2034, 2034, 2059, 124, 1010, 2345, 1234567788, 123456, 910, 910, 10.34, 1234556], 'Strings': ['alo23', 'natalia.batista', 'msudid', 1234556, '4sss', '4sss', '4sss', '4sss', 'aline.santos', 'anna', 'natalia.batista', 'joao.pedro', 'maria.clara']}
{'Numeros': [123456, 1234556, 2345, 10.34, 2059, 1234567788, 910, 1010, 2034, 124], 'Strings': ['anna', '4sss', 'alo23', 'msudid', 'aline.santos', 'natalia.batista', 'maria.clara', 1234556, 'joao.pedro']}


## Fase 2

In [318]:
print(dicionario)

{'Numeros': [123456, 1234556, 2345, 10.34, 2059, 1234567788, 910, 1010, 2034, 124], 'Strings': ['anna', '4sss', 'alo23', 'msudid', 'aline.santos', 'natalia.batista', 'maria.clara', 1234556, 'joao.pedro']}


Filtrando e formatando os dados com as funções map() e filter(), com a aplicação da função lambda

In [319]:
#removendo os numeros que ficaram na lista de strings
dicionario['Strings'] = list(filter(lambda x: isinstance(x, str), dicionario['Strings']))
print(dicionario)
#removendo numeros com menos de 6 dígitos
dicionario['Numeros'] = list(filter(lambda x: len(str(x))>=6, dicionario['Numeros']))
print(dicionario)
#padronizando enderecos de email, baseado no usuario
dicionario['Strings'] = list(map(lambda x: x+"@inatel.br", dicionario['Strings']))
print(dicionario)

{'Numeros': [123456, 1234556, 2345, 10.34, 2059, 1234567788, 910, 1010, 2034, 124], 'Strings': ['anna', '4sss', 'alo23', 'msudid', 'aline.santos', 'natalia.batista', 'maria.clara', 'joao.pedro']}
{'Numeros': [123456, 1234556, 1234567788], 'Strings': ['anna', '4sss', 'alo23', 'msudid', 'aline.santos', 'natalia.batista', 'maria.clara', 'joao.pedro']}
{'Numeros': [123456, 1234556, 1234567788], 'Strings': ['anna@inatel.br', '4sss@inatel.br', 'alo23@inatel.br', 'msudid@inatel.br', 'aline.santos@inatel.br', 'natalia.batista@inatel.br', 'maria.clara@inatel.br', 'joao.pedro@inatel.br']}


OBS: Por se tratar de classes, o tratamento de erros foi inserido diretamente na fase 3

## Fase 3

O Pipeline está organizado da seguinta maneira:

```
SeparacaoDados
      |
      v
TratamentoDados (Abstrata)
      |
      v
TratamentoGeral
|             |
Numeros       Strings      
```




In [320]:
class SeparacaoDados:
  """
  Converte os dados em um dicionário, separando-os em 'Numeros' e 'Strings'
  Essa etapa é importante para separar as matrículas dos usuários
  """
  def __init__(self, input_text):
    self.dicionario = {'Numeros': [], 'Strings': []}

    for item in input_text:
        if isinstance(item, int):
            self.dicionario['Numeros'].append(item)
        else:
            self.dicionario['Strings'].append(item)

In [321]:
from abc import ABC, abstractmethod

class TratamentoDados(ABC):
  """
  Classe abstrata que formaliza a obrigatoriedade das classes calibracao e limpeza em todos os saneadores
  """
  @abstractmethod
  def calibracao(self):
    pass


  @abstractmethod
  def limpeza(self):
    pass

In [322]:
class TratamentoGeral(TratamentoDados):
  """
  Herda da classe abstrata TratamentoDados
   """
  def __init__(self,dicionario):
     self.dicionario = dicionario


  def remocao_duplicatas(self, lista):
    """utiliza set para remover valores duplicados"""
    lista = set(lista)
    return list(lista)


Classe que executa o tratamento dos 'Numeros' (matrículas)

In [323]:
class TratamentoNumeros(TratamentoGeral):
  """
  Saneador para tratamento específico dos valores numéricos (matrículas)
  Remove valores duplicados e não aceita matrículas com mais de 4 dígitos
  """
  def __init__(self, dicionario):
     super().__init__(dicionario)


  def calibracao(self):
    """Garante que os números de matrícula são inteiros"""
    numeros_calibrados = []
    for item in self.dicionario['Numeros']:
      if isinstance(item, int):
          numeros_calibrados.append(item)
      elif isinstance(item, float):
          numeros_calibrados.append(int(item))

    self.dicionario['Numeros'] = numeros_calibrados


  def limpeza(self):
    """Remove números com mais de 4 dígitos"""
    self.dicionario['Numeros'] = list(filter(lambda x: len(str(x))<=4, self.dicionario['Numeros']))


  def remocao_duplicatas(self):
    self.dicionario['Numeros'] = super().remocao_duplicatas(self.dicionario['Numeros'])

Classe que executa o tratamento das 'Strings' (usuários) e as formata como e-mail

In [324]:
import unicodedata

In [325]:
class TratamentoStrings(TratamentoGeral):
  """
  Saneador para tratamento específico de strings (usuários)
  Além de efetuar manipulações básicas de strings, padroniza-as no formato de email do inatel
  """
  def __init__(self, dicionario):
     super().__init__(dicionario)

  def reclassificacao_numeros(self, item):
    """ Adiciona o item encontrado na categoria 'Numeros' do dicionário (faz uma reclassificacao)"""
    self.dicionario['Numeros'].append(item)


  def calibracao(self):
    nova_lista = []
    """Verifica se há numeros inseridos como strings.
     Caso haja, os converte para inteiros e chama a reclassificacao.
     Mantém apenas os elementos do tipo 'str' em self.dicionario['Strings'']
     """

    for i in range(len(self.dicionario['Strings'])):
      try:
        self.dicionario['Strings'][i] = int(self.dicionario['Strings'][i])
        self.reclassificacao_numeros(self.dicionario['Strings'][i])
      except ValueError:
        nova_lista.append(self.dicionario['Strings'][i])

    self.dicionario['Strings'] = nova_lista

  @staticmethod
  def limpar_caracteres(texto):
    """Remove caracteres que não são alfanuméricos da string """
    return "".join(
        caractere for caractere in texto if caractere.isalnum() or caractere == '.'
    )

  @staticmethod
  def remover_acentos(texto):
    """Remove acentos para padronizar como email"""
    texto_normalizado = unicodedata.normalize('NFD', texto)
    return ''.join(c for c in texto_normalizado if unicodedata.category(c) != 'Mn')


  def limpeza(self):
    """
    Percorre a lista de Strings do dicionário
    Remove espacos extras, padroniza o texto como caixa baixa e remove caracteres especiais das strings
    """

    for i in range(len(self.dicionario['Strings'])):
      print(type(self.dicionario['Strings'][i]))
      self.dicionario['Strings'][i] = self.dicionario['Strings'][i].strip()
      self.dicionario['Strings'][i] = self.dicionario['Strings'][i].lower()
      self.dicionario['Strings'][i] = self.limpar_caracteres(self.dicionario['Strings'][i])
      self.dicionario['Strings'][i] = self.remover_acentos(self.dicionario['Strings'][i])



  def remocao_duplicatas(self):
    """
    Chama o método da classe mãe que remove as duplicatas na lista de strings.
    Adicionalmente, remove os numeros que permaneceram na lista de strings
    """
    self.dicionario['Strings'] = super().remocao_duplicatas(self.dicionario['Strings'])
    self.dicionario['Strings'] = list(filter(lambda x: isinstance(x, str), self.dicionario['Strings']))


  def padronizando_emails(self):
    """ padroniza os dados de usuario para o formato de email do inatel usando a funcao map()"""
    self.dicionario['Strings'] = list(map(lambda x: x+"@inatel.br", self.dicionario['Strings']))

In [326]:
import pandas as pd

Exceptions Personalizadas para o saneador

In [327]:
class ErroLista(Exception):
  """
  Raised se os dados de entrada não são uma lista
  """
  pass

In [328]:
class EstouroMemoria(Exception):
  """
  Raised se algum elemento da lista de entrada possui mais de 100 caracteres
  Como o saneador assume que os dados são usuários e matrículas, não faz sentido aceitar elementos > 100 caracteres
  """
  pass

Descritor usado para a verificação dos dados de entrada. Verifica se os dados seguem o padrão desejado

In [329]:
class VerificadorDados:

  def __get__(self, obj, owner):
      return obj.__dict__.get("dados")

  def __set__(self, obj, valor):

      if not isinstance(valor, list):
          raise ErroLista("'dados' deve ser uma lista")

      for item in valor:
          if isinstance(item, str) and len(item) > 100:
              raise EstouroMemoria("String com mais de 100 caracteres encontrada")
          if isinstance(item, int) and len(str(item)) > 100:
              raise EstouroMemoria("Valor numérico com mais de 100 caracteres encontrado")

      obj.__dict__["dados"] = valor

Classe que sumariza todo o Pipeline. Ela encadeia todas as etapas, recebendo os dados brutos e retornando um DataFrame do Pandas que poderá ser utilizado por outras bibliotecas. Como o intuito do código é apenas tratar os dados, e a entrada não estabelece uma relação direta entre o usuário e a matrícula, o DataFrame pode conter valores NaN

In [330]:
class PipelineSASD:
  """
  Classe que recebe os dados brutos e estrutura o Pipeline de processamento.
  Une todos os passos a fim de gerar o DataFrame final
  """
  dados = VerificadorDados()
  def __init__(self, dados):
    """Instancia SeparacaoDados, TratamentoNumeros, TratamentoStrings"""
    self.dados = dados #O descritor irá analisar essa atribuição
    separador = SeparacaoDados(dados)

    self.dicionario = separador.dicionario
    self.trat_strings = TratamentoStrings(self.dicionario)
    self.trat_numeros = TratamentoNumeros(self.dicionario)

  def executar(self):
    #Realiza todo o processamento das Strings (Usuários)
    self.trat_strings.calibracao()
    self.trat_strings.limpeza()
    self.trat_strings.remocao_duplicatas()
    self.trat_strings.padronizando_emails()
    #Realiza o tratamento dos números (matrículas)
    self.trat_numeros.limpeza()
    self.trat_numeros.remocao_duplicatas()

    return pd.DataFrame(dict([(k,pd.Series(v)) for k,v in self.dicionario.items()]))

  @classmethod
  def exemplo(cls):
    """Usado sem necessidade de instanciação, o método gera uma pequena base de dados para teste do pipeline  """
    dados = ["  João  ", "MARIA", "123456", "abc@", 9876,"000123"]
    return cls(dados)




Testes do Pipeline

In [331]:
pipeline = PipelineSASD(input_text)

resultado = pipeline.executar()

print(resultado)

<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
   Numeros                    Strings
0   2345.0             anna@inatel.br
1     10.0             4sss@inatel.br
2   2059.0            alo23@inatel.br
3    910.0           msudid@inatel.br
4   1010.0  natalia.batista@inatel.br
5   2034.0      maria.clara@inatel.br
6    124.0     aline.santos@inatel.br
7      NaN       joao.pedro@inatel.br


In [332]:
pipeline = PipelineSASD.exemplo()
resultado = pipeline.executar()
print(resultado)

<class 'str'>
<class 'str'>
<class 'str'>
   Numeros          Strings
0    123.0  maria@inatel.br
1   9876.0    abc@inatel.br
2      NaN   joao@inatel.br


Erros gerados pelo descritor

```
EstouroMemoria: String com mais de 100 caracteres encontrada
```



In [333]:
dados = ["a"*120]

p = PipelineSASD(dados)
resultado = p.executar()
print(resultado)

EstouroMemoria: String com mais de 100 caracteres encontrada

```
EstouroMemoria: Valor numérico com mais de 100 caracteres encontrado
```

In [334]:
dados = [2**1000]

p = PipelineSASD(dados)
resultado = p.executar()
print(resultado)

EstouroMemoria: Valor numérico com mais de 100 caracteres encontrado